# CXR Intelligence — Full Pipeline (Kaggle, T4×2, free tier)

**DSAI 413 Assignment 2** — runs end-to-end on a single Kaggle session with T4×2 GPU. Reproduces every artifact in the repo.

## Prerequisites
1. Kaggle notebook with **Settings → Accelerator = GPU T4 ×2**
2. **+ Add Input** → search `simhadrisadaram/mimic-cxr-dataset` → add
3. **Add-ons → Secrets** → add these (toggle Notebook access ON for each):
   - `HF_TOKEN` (accept the [MedGemma TOS](https://huggingface.co/google/medgemma-4b-it) first)
   - `NVIDIA_API_KEY` (from https://build.nvidia.com)
   - `KAGGLE_USERNAME`, `KAGGLE_KEY` (from https://www.kaggle.com/settings)
   - `NGROK_TOKEN` (optional, for public demo URL — get free at https://ngrok.com)

## Stages (run in order)
1. Bootstrap — clone repo, install deps, symlink dataset, load secrets
2. Preprocess — extract reports, CheXpert labels, train/val/test splits (~3 min)
3. QA dataset — synthesize via NVIDIA NIM (~90 min, 7000+ pairs)
4. Build indices — BiomedCLIP + ColPali (with patched adapter, ~25 min)
5. Evaluate — 3 configs × 3 modes (~80 min)
6. (Optional) Gradio + ngrok demo for clickable public URL

## Stage 1 — Bootstrap

In [ ]:
import os, sys, pathlib, subprocess

# Clone repo (if not present) and install
if not pathlib.Path('/kaggle/working/cxr-intel').exists():
    subprocess.run(['git', 'clone', 'https://github.com/jo3591/assigment2dsai413', '/kaggle/working/cxr-intel'], check=True)
%cd /kaggle/working/cxr-intel
!git pull
!pip install -q -r requirements-colab.txt
!pip install -q -e .

# Critical dep fix: peft 0.15+ requires triton.ops which conflicts; uninstall bitsandbytes (we'll reinstall a compatible version later for MedGemma)
!pip install -q --force-reinstall --no-deps 'transformers>=4.50,<4.53'

# Make package importable
sys.path.insert(0, '/kaggle/working/cxr-intel/src')

# Load Kaggle Secrets into env
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for k in ['HF_TOKEN','NVIDIA_API_KEY','KAGGLE_USERNAME','KAGGLE_KEY']:
    try: os.environ[k] = secrets.get_secret(k)
    except Exception: pass
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['QA_SYNTH_MODEL'] = 'meta/llama-3.1-8b-instruct'
os.environ['QA_JUDGE_MODEL'] = 'meta/llama-3.3-70b-instruct'

# Symlink the Kaggle-attached dataset into our expected path
dst = pathlib.Path('/kaggle/working/cxr-intel/data/raw')
dst.parent.mkdir(parents=True, exist_ok=True)
if not dst.exists():
    dst.symlink_to('/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset')

import torch
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
print(f'data/raw: {[p.name for p in dst.iterdir()][:3]}')

## Stage 2 — Preprocess reports (~3 min)

Loads `mimic_cxr_aug_train.csv`, parses Findings/Impression, computes rule-based CheXpert labels, builds patient-level train/val/test splits.

In [ ]:
!python -u scripts/download_data.py --config configs/data.yaml --skip-download --no-basename-index

## Stage 3 — Build QA dataset (~90 min, NVIDIA NIM)

Synthesizes ~7000 QA pairs across 5 question types (existence/location/severity/attribute/open), anchored on CheXpert labels. Validator drops ~10% for banned terms or source-mismatch.

In [ ]:
!python -u scripts/build_qa_dataset.py --config configs/data.yaml

## Stage 4 — Build retrieval indices (~30 min on T4)

### 4a. BiomedCLIP (5 min)

In [ ]:
!python -u scripts/build_indices.py --backend biomedclip

### 4b. ColPali — patch the v1.3 LoRA adapter for transformers 5.x

Newer transformers refactored PaliGemma's layer paths (`model.language_model.model.X` → `model.model.language_model.X`). The official `vidore/colpali-v1.3` adapter is keyed against the old paths, so we patch it offline so the LoRA weights actually merge.

In [ ]:
import shutil, safetensors.torch
from huggingface_hub import snapshot_download

full_src = pathlib.Path(snapshot_download('vidore/colpali-v1.3'))
patched = pathlib.Path('/kaggle/working/colpali-v1.3-patched')
if patched.exists():
    shutil.rmtree(patched)
patched.mkdir(parents=True)
for src in full_src.iterdir():
    if src.is_file():
        shutil.copy2(src, patched / src.name)

# Remap adapter keys
state = safetensors.torch.load_file(str(patched / 'adapter_model.safetensors'))
remapped = {k.replace('model.language_model.model.', 'model.model.language_model.'): v for k, v in state.items()}
safetensors.torch.save_file(remapped, str(patched / 'adapter_model.safetensors'))
print(f'✓ Patched {len(remapped)} adapter keys → {patched}')

### 4c. Build ColPali index using the patched checkpoint (~25 min)

In [ ]:
import pandas as pd
from cxr_intel.retrieval.colpali_index import ColPaliRetriever

df = pd.read_parquet('data/processed/reports.parquet')
items = [
    {'study_id': str(r['study_id']),
     'image_path': str(r['image_path']),
     'report_text': (str(r.get('findings','')) + ' ' + str(r.get('impression',''))).strip()}
    for _, r in df.iterrows()
]
print(f'Indexing {len(items)} items with PATCHED ColPali...')

r = ColPaliRetriever(
    checkpoint='/kaggle/working/colpali-v1.3-patched',
    torch_dtype='float16',
    device_map='auto',
    batch_size=2,
)
r.index(items, 'data/indices/colpali_zs')
print(f'✓ Done. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Stage 5 — Run full evaluation (~80 min, T4)

**Critical: restart the kernel between Stage 4 and Stage 5** to free VRAM held by the ColPali model used for indexing. Then re-run Stage 1 (Bootstrap) and skip to Stage 5.

### 5a. Patch the eval script to use INT4 MedGemma + the patched ColPali path

In [ ]:
# Install bnb >=0.45 for INT4 MedGemma quantization (handles newer triton)
!pip install -q --upgrade 'bitsandbytes>=0.45,<0.50'

# Patch run_eval.py to use int4 MedGemma + the patched ColPali checkpoint
p = pathlib.Path('scripts/run_eval.py')
content = p.read_text()
new = content.replace(
    'medgemma = MedGemmaRunner()\n    medgemma.load()',
    "medgemma = MedGemmaRunner(quantization='int4')\n    medgemma.load()"
).replace(
    'r = ColPaliRetriever(name="colpali_zs")',
    'r = ColPaliRetriever(name="colpali_zs", checkpoint="/kaggle/working/colpali-v1.3-patched")',
)
p.write_text(new)

# Patch metrics_report.py for newer sacrebleu BLEU class API
mr = pathlib.Path('src/cxr_intel/eval/metrics_report.py')
mrc = mr.read_text()
if 'BLEU(max_ngram_order' not in mrc:
    mrc = mrc.replace(
        'import sacrebleu\n\n    bleu1 = sacrebleu.corpus_bleu(preds, [refs], max_ngram_order=1).score / 100',
        'from sacrebleu.metrics import BLEU\n\n    bleu1 = BLEU(max_ngram_order=1, effective_order=True).corpus_score(preds, [refs]).score / 100'
    ).replace(
        'bleu2 = sacrebleu.corpus_bleu(preds, [refs], max_ngram_order=2).score / 100',
        'bleu2 = BLEU(max_ngram_order=2, effective_order=True).corpus_score(preds, [refs]).score / 100'
    ).replace(
        'bleu4 = sacrebleu.corpus_bleu(preds, [refs]).score / 100',
        'bleu4 = BLEU(effective_order=True).corpus_score(preds, [refs]).score / 100'
    )
    mr.write_text(mrc)
print('✓ Patches applied')

### 5b. Smoke test (5 studies, ~5 min) — verify the pipeline before committing to 80 min

In [ ]:
!python -u scripts/run_eval.py --mode report --configs medgemma_only,biomedclip_rag,colpali_zs_rag --test-size 5 --top-k 3

In [ ]:
import pandas as pd
pd.read_csv('results/tables/report_metrics.csv')

### 5c. Full eval (50 studies × 3 configs × 3 modes, ~80 min)

Writes:
- `results/tables/report_metrics.csv`
- `results/tables/qa_metrics.csv`
- `results/tables/retrieval_metrics.csv`
- `results/predictions/report.json`, `results/predictions/qa.json`

In [ ]:
!python -u scripts/run_eval.py \
    --mode report --mode qa --mode retrieval \
    --configs medgemma_only,biomedclip_rag,colpali_zs_rag \
    --test-size 50 --top-k 3

In [ ]:
# Inspect the final results
import pandas as pd
for name in ['report_metrics', 'qa_metrics', 'retrieval_metrics']:
    print(f'\n=== {name} ===')
    print(pd.read_csv(f'results/tables/{name}.csv').to_string(index=False))

## Stage 6 — (Optional) Gradio + ngrok demo

See `notebooks/07_gradio_demo_kaggle.ipynb` — open it as a separate Kaggle notebook (or paste its cells into a new section here). Produces a clickable public URL via ngrok.

## Save Version (Quick Save) to persist all outputs to Kaggle

Top-right → **Save Version** → **Quick Save** → name e.g. `cxr-full-pipeline-v1`. This commits `/kaggle/working/` as a notebook Output that's downloadable and re-attachable.

## What's in this notebook's output (after Quick Save)
- `data/processed/reports.parquet` — preprocessed corpus
- `data/processed/splits.json` — patient-level train/val/test
- `data/qa/qa_v1.jsonl` + `data/qa/qa_test.jsonl` — synthesized QA dataset
- `data/indices/biomedclip/` + `data/indices/colpali_zs/` — retrieval indices
- `colpali-v1.3-patched/` — patched ColPali adapter
- `results/tables/*.csv` — all evaluation metrics
- `results/predictions/*.json` — per-test-case predictions